# Fase 2 — Evaluación y adaptación del modelo para el teléfono
**Tesis:** Desarrollo de una aplicación móvil para el diagnóstico automatizado de la enfermedad ojo de gallo del cafeto (*Mycena citricolor*) mediante visión por computadora con inferencia offline.
**Autor:** Rodrigo Manuel Alfaro Giraldo — Universidad Cooperativa de Colombia.

Este cuaderno hace, en orden:
1. Verifica que los datos y el modelo estén completos.
2. Reproduce la **línea base** del modelo `mejor_modelo.keras` (objetivo 1).
3. Evalúa el modelo con un **conjunto de prueba depurado** de imágenes casi duplicadas.
4. Genera mapas **Grad-CAM** para ver en qué parte de la foto se fija el modelo.
5. Entrena el **modelo verificador** "¿es una hoja de café?" (Fase 2).
6. Evalúa el **flujo completo**: verificador + diagnóstico + resultado no concluyente.
7. Convierte ambos modelos a **TensorFlow Lite** (4 variantes) y mide el efecto de la conversión (objetivo 2).
8. Guarda todos los resultados (tablas, figuras y modelos) en tu Google Drive.

**Cómo usarlo:** menú *Entorno de ejecución → Ejecutar todo*. No cambies nada salvo la celda de **Configuración** si tus carpetas tienen otros nombres.

## 0. Instalación de librerías
Se ejecuta una sola vez. Si Colab pide **reiniciar la sesión**, acepta y vuelve a *Ejecutar todo*.

In [1]:
!pip -q install -U "keras>=3.15.1" imagehash ai-edge-litert

## 1. Configuración
Estructura esperada en tu Google Drive (igual a la de tu computador):
```
MyDrive/tesis_ojo_de_gallo/
├── mejor_modelo.keras
├── 02_Dataset/
│   ├── Ojo_Gallo/   (500 fotos: 20251129_091427.jpg, ...)
│   └── Sanas/       (499 fotos: Sana 1.jpg, ...)
└── negativos_propios/   (OPCIONAL: tus fotos de manos, objetos, suelo y hojas que NO son de café)
```
El cuaderno arma solo las carpetas de entrenamiento, validación y prueba con la **partición original** del modelo
(la misma que produjo el resultado de 151/151), guardada en la celda siguiente.

In [2]:
import os, sys, json, time, math, random, shutil, warnings
warnings.filterwarnings("ignore")

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

RUTA_BASE = os.environ.get("RUTA_BASE", "/content/drive/MyDrive/tesis_ojo_de_gallo")
RUTA_FOTOS = {"ojo_de_gallo": os.path.join(RUTA_BASE, "02_Dataset", "Ojo_Gallo"),
              "sana": os.path.join(RUTA_BASE, "02_Dataset", "Sanas")}
RUTA_MODELO = os.path.join(RUTA_BASE, "mejor_modelo.keras")
RUTA_NEGATIVOS_PROPIOS = os.path.join(RUTA_BASE, "negativos_propios")
RUTA_SALIDA = os.path.join(RUTA_BASE, "fase2_resultados")
RUTA_DATASET = os.environ.get("RUTA_DATASET", "/content/dataset_modelo")   # copia local, más rápida que leer desde Drive
EXT = (".jpg", ".jpeg", ".png", ".bmp")

SEMILLA = 42
TAM = 224                      # tamaño de entrada del modelo (224 x 224)
CLASES = ["ojo_de_gallo", "sana"]
UMBRAL = 0.5                   # probabilidad de ojo de gallo a partir de la cual se diagnostica la enfermedad
BANDA_NO_CONCLUYENTE = (0.30, 0.70)  # si la probabilidad cae en este rango, el resultado es "no concluyente" (criterio del autor)
UMBRAL_VERIFICADOR = 0.5       # probabilidad mínima de "hoja de café" para aceptar la imagen
UMBRALES_DEPURACION = [0, 2, 4]  # distancias de hash perceptual para depurar el conjunto de prueba
N_NEGATIVOS_POR_FUENTE = int(os.environ.get("N_NEG", 400))
EPOCAS_VERIFICADOR = int(os.environ.get("EPOCAS_VER", 15))

for sub in ["tablas", "figuras", "modelos", "android"]:
    os.makedirs(os.path.join(RUTA_SALIDA, sub), exist_ok=True)

random.seed(SEMILLA)
import numpy as np
np.random.seed(SEMILLA)
import tensorflow as tf
tf.random.set_seed(SEMILLA)
import keras
import pandas as pd
import matplotlib
matplotlib.use("Agg") if not EN_COLAB else None
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import imagehash
from sklearn.metrics import roc_auc_score, roc_curve

VERSIONES = {"python": sys.version.split()[0], "tensorflow": tf.__version__, "keras": keras.__version__,
             "numpy": np.__version__, "gpu": [d.name for d in tf.config.list_physical_devices("GPU")]}
print(json.dumps(VERSIONES, indent=2))

def guardar_json(obj, nombre):
    with open(os.path.join(RUTA_SALIDA, "tablas", nombre), "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=float)

Mounted at /content/drive
{
  "python": "3.13.16",
  "tensorflow": "2.21.0",
  "keras": "3.15.1",
  "numpy": "2.1.3",
  "gpu": [
    "/physical_device:GPU:0"
  ]
}


## 2. Partición original y verificación de datos
Se copian las fotos a `train`, `validation` y `test` según la partición original del modelo.
Los conteos deben coincidir con la Tabla 4 de la tesis: 350/349, 75/74 y 75/76.

In [3]:
PARTICION_ORIGINAL = json.loads(r'''{"train": {"ojo_de_gallo": ["20251129_091959.jpg", "20251129_092016.jpg", "20251129_092053.jpg", "20251129_092106.jpg", "20251129_092123.jpg", "20251129_092139.jpg", "20251129_092149.jpg", "20251129_092158.jpg", "20251129_092220.jpg", "20251129_092252.jpg", "20251129_092438.jpg", "20251129_092457.jpg", "20251129_092504.jpg", "20251129_092511.jpg", "20251129_092539.jpg", "20251129_092559.jpg", "20251129_092702.jpg", "20251129_092708.jpg", "20251129_092800.jpg", "20251129_092931.jpg", "20251129_093055.jpg", "20251129_093132.jpg", "20251129_093207.jpg", "20251129_093218.jpg", "20251129_093257.jpg", "20251129_093304.jpg", "20251129_093324.jpg", "20251129_093359.jpg", "20251129_093502.jpg", "20251129_093615.jpg", "20251129_093623.jpg", "20251129_093725.jpg", "20251129_093729.jpg", "20251129_093736.jpg", "20251129_093956.jpg", "20251129_094006.jpg", "20251129_094032.jpg", "20251129_094102.jpg", "20251129_094107.jpg", "20251129_094130.jpg", "20251129_094219.jpg", "20251129_094307.jpg", "20251129_094337.jpg", "20251129_094614.jpg", "20251129_094622.jpg", "20251129_094627.jpg", "20251129_094654.jpg", "20251129_094701.jpg", "20251129_094834.jpg", "20251129_094950.jpg", "20251129_095011.jpg", "20251129_095106.jpg", "20251129_095132.jpg", "20251129_095138.jpg", "20251129_095206.jpg", "20251129_095244.jpg", "20251129_095331.jpg", "20251129_095408.jpg", "20251129_095416.jpg", "20251129_095449.jpg", "20251129_095512.jpg", "20251129_095538.jpg", "20251129_095552.jpg", "20251129_095556.jpg", "20251129_095601.jpg", "20251129_095609.jpg", "20251129_095629.jpg", "20251129_095642.jpg", "20251129_095712.jpg", "20251129_095729.jpg", "20251129_095736.jpg", "20251129_095755.jpg", "20251129_095810.jpg", "20251129_095821.jpg", "20251129_100046.jpg", "20251129_100416.jpg", "20251129_100436.jpg", "20251129_100506.jpg", "20251129_100515.jpg", "20251129_100535.jpg", "20251129_100547.jpg", "20251129_100621.jpg", "20251129_100711.jpg", "20251129_100808.jpg", "20251129_100906.jpg", "20251129_100918.jpg", "20251129_100948.jpg", "20251129_100955.jpg", "20251129_101023.jpg", "20251129_101029.jpg", "20251129_101033.jpg", "20251129_101157.jpg", "20251129_101255.jpg", "20251129_101307.jpg", "20251129_101319.jpg", "20251129_101337.jpg", "20251129_101342.jpg", "20251129_101430.jpg", "20251129_101441.jpg", "20251129_101504.jpg", "20251129_101509.jpg", "20251129_101518.jpg", "20251129_101529.jpg", "20251129_101559.jpg", "20251129_101610.jpg", "20251129_101624.jpg", "20251129_101641.jpg", "20251129_101714.jpg", "20251129_101831.jpg", "20251129_101839.jpg", "20251129_101856.jpg", "20251129_101905.jpg", "20251129_102019.jpg", "20251129_102125.jpg", "20251129_102402.jpg", "20251129_102518.jpg", "20251129_102657.jpg", "20251129_102737.jpg", "20251129_102838.jpg", "20251129_102905.jpg", "20251129_103003.jpg", "20251129_103051.jpg", "20251129_103124.jpg", "20251129_103139.jpg", "20251129_103154.jpg", "20251129_103206.jpg", "20251129_103304.jpg", "20251129_103311.jpg", "20251129_103353.jpg", "20251129_103409.jpg", "20251129_103437.jpg", "20251129_103445.jpg", "20251129_103532.jpg", "20251129_103542.jpg", "20251129_103604.jpg", "20251129_103618.jpg", "20251129_103647.jpg", "20251129_103728.jpg", "20251129_103758.jpg", "20251129_103804.jpg", "20251129_103839.jpg", "20251129_103847.jpg", "20251129_103938.jpg", "20251129_104212.jpg", "20251129_104314.jpg", "20251129_104320.jpg", "20251129_104346.jpg", "20251129_104351.jpg", "20251129_104356.jpg", "20251129_104409.jpg", "20251129_104557.jpg", "20251129_104706.jpg", "20251129_104727.jpg", "20251129_104745.jpg", "20251129_104803.jpg", "20251129_104901.jpg", "20251129_104924.jpg", "20251129_104958.jpg", "20251129_105257.jpg", "20251129_105404.jpg", "20251129_105435.jpg", "20251129_105505.jpg", "20251129_105517.jpg", "20251129_105537.jpg", "20251129_114546.jpg", "20251129_114556.jpg", "20251129_114630.jpg", "20251129_114655.jpg", "20251129_114830.jpg", "20251129_114858.jpg", "20251129_114932.jpg", "20251129_114957.jpg", "20251129_115010.jpg", "20251129_115017.jpg", "20251129_115123.jpg", "20251129_115140.jpg", "20251129_115201.jpg", "20251129_115218.jpg", "20251129_115223.jpg", "20251129_115251.jpg", "20251129_115305.jpg", "20251129_115313.jpg", "20251129_115322.jpg", "20251129_115331.jpg", "20251129_115426.jpg", "20251129_115503.jpg", "20251129_115534.jpg", "20251129_115633.jpg", "20251129_115719.jpg", "20251129_115734.jpg", "20251129_115754.jpg", "20251129_115815.jpg", "20251129_115822.jpg", "20251129_115827.jpg", "20251129_115900.jpg", "20251129_115923.jpg", "20251129_120045.jpg", "20251129_120152.jpg", "20251129_120217.jpg", "20251129_120227.jpg", "20251129_120259.jpg", "20251129_120417.jpg", "20251129_120512.jpg", "20251129_120518.jpg", "20251129_120531.jpg", "20251129_120544.jpg", "20251129_120552.jpg", "20251129_120745.jpg", "20251129_120748.jpg", "20251129_120803.jpg", "20251129_120820.jpg", "20251129_120831.jpg", "20251129_120901.jpg", "20251129_120915.jpg", "20251129_120956.jpg", "20251129_121017.jpg", "20251129_121030.jpg", "20251129_121051.jpg", "20251129_121134.jpg", "20251129_130238.jpg", "20251129_130306.jpg", "20251129_130402.jpg", "20251129_130457.jpg", "20251129_130522.jpg", "20251129_130644.jpg", "20251129_130715.jpg", "20251129_130723.jpg", "20251129_130733.jpg", "20251129_130802.jpg", "20251129_130807.jpg", "20251129_130817.jpg", "20251129_130850.jpg", "20251129_131040.jpg", "20251129_131046.jpg", "20251129_131056.jpg", "20251129_131125.jpg", "20251129_131202.jpg", "20251129_131320.jpg", "20251129_131514.jpg", "20251129_131521.jpg", "20251129_131525.jpg", "20251129_131529.jpg", "20251129_131655.jpg", "20251129_131720.jpg", "20251129_131750.jpg", "20251129_131815.jpg", "20251129_131820.jpg", "20251129_131839.jpg", "20251129_131901.jpg", "20251129_131914.jpg", "20251129_131938.jpg", "20251129_132038.jpg", "20251129_132125.jpg", "20251129_132200.jpg", "20251129_132354.jpg", "20251129_132449.jpg", "20251129_132501.jpg", "20251129_132528.jpg", "20251129_132534.jpg", "20251129_132732.jpg", "20251129_132751.jpg", "20251129_132832.jpg", "20251129_132854.jpg", "20251129_132915.jpg", "20251129_132921.jpg", "20251129_132954.jpg", "20251129_133000.jpg", "20251129_133018.jpg", "20251129_133340.jpg", "20251129_133353.jpg", "20251129_133423.jpg", "20251129_133456.jpg", "20251129_133515.jpg", "20251129_133519.jpg", "20251129_133625.jpg", "20251129_133633.jpg", "20251129_133740.jpg", "20251129_133814.jpg", "20251129_133850.jpg", "20251129_133912.jpg", "20251129_133953.jpg", "20251129_133959.jpg", "20251129_134106.jpg", "20251129_134335.jpg", "20251129_134401.jpg", "20251129_134420.jpg", "20251129_134502.jpg", "20251129_134506.jpg", "20251129_134517.jpg", "20251129_134527.jpg", "20251129_134557.jpg", "20251129_134615.jpg", "20251129_134713.jpg", "20251129_134724.jpg", "20251129_135000.jpg", "20251129_135004.jpg", "20251129_135020.jpg", "20251129_135027.jpg", "20251129_135036.jpg", "20251129_135042.jpg", "20251129_135118.jpg", "20251129_135142.jpg", "20251129_135152.jpg", "20251129_135205.jpg", "20251129_135212.jpg", "20251129_135217.jpg", "20251129_135225.jpg", "20251129_135235.jpg", "20251129_135253.jpg", "20251129_135335.jpg", "20251129_135341.jpg", "20251129_135422.jpg", "20251129_135428.jpg", "20251129_135448.jpg", "20251129_135510.jpg", "20251129_135538.jpg", "20251129_135551.jpg", "20251129_135556.jpg", "20251129_135755.jpg", "20251129_135804.jpg", "20251129_135827.jpg", "20251129_135845.jpg", "20251129_140008.jpg", "20251129_140011.jpg", "20251129_140020.jpg", "20251129_140038.jpg", "20251129_140128.jpg", "20251129_140136.jpg", "20251129_140207.jpg", "20251129_140216.jpg", "20251129_140227.jpg", "20251129_140240.jpg", "20251129_140502.jpg", "20251129_140505.jpg", "20251129_140513.jpg", "20251129_140528.jpg", "20251129_140538.jpg", "20251129_140553.jpg", "20251129_140618.jpg", "20251129_140653.jpg", "20251129_140657.jpg", "20251129_140708.jpg", "20251129_140713.jpg", "20251129_140744.jpg", "20251129_140816.jpg", "20251129_140820.jpg", "20251129_140842.jpg", "20251129_140850.jpg", "20251129_140907.jpg", "20251129_140938.jpg"], "sana": ["Sana 1.jpg", "Sana 100.jpg", "Sana 103.jpg", "Sana 106.jpg", "Sana 107.jpg", "Sana 109.jpg", "Sana 110.jpg", "Sana 111.jpg", "Sana 114.jpg", "Sana 115.jpg", "Sana 118.jpg", "Sana 121.jpg", "Sana 124.jpg", "Sana 125.jpg", "Sana 127.jpg", "Sana 129.jpg", "Sana 131.jpg", "Sana 138.jpg", "Sana 14.jpg", "Sana 141.jpg", "Sana 143.jpg", "Sana 147.jpg", "Sana 148.jpg", "Sana 15.jpg", "Sana 154.jpg", "Sana 156.jpg", "Sana 163.jpg", "Sana 169.jpg", "Sana 173.jpg", "Sana 178.jpg", "Sana 18.jpg", "Sana 183.jpg", "Sana 188.jpg", "Sana 192.jpg", "Sana 194.jpg", "Sana 199.jpg", "Sana 20.jpg", "Sana 201.jpg", "Sana 203.jpg", "Sana 205.jpg", "Sana 206.jpg", "Sana 207.jpg", "Sana 208.jpg", "Sana 214.jpg", "Sana 221.jpg", "Sana 222.jpg", "Sana 223.jpg", "Sana 224.jpg", "Sana 225.jpg", "Sana 227.jpg", "Sana 237.jpg", "Sana 240.jpg", "Sana 241.jpg", "Sana 243.jpg", "Sana 244.jpg", "Sana 245.jpg", "Sana 249.jpg", "Sana 251.jpg", "Sana 253.jpg", "Sana 254.jpg", "Sana 255.jpg", "Sana 258.jpg", "Sana 259.jpg", "Sana 260.jpg", "Sana 261.jpg", "Sana 264.jpg", "Sana 267.jpg", "Sana 268.jpg", "Sana 269.jpg", "Sana 27.jpg", "Sana 272.jpg", "Sana 276.jpg", "Sana 278.jpg", "Sana 288.jpg", "Sana 29.jpg", "Sana 293.jpg", "Sana 294.jpg", "Sana 295.jpg", "Sana 307.jpg", "Sana 311.jpg", "Sana 313.jpg", "Sana 32.jpg", "Sana 323.jpg", "Sana 332.jpg", "Sana 339.jpg", "Sana 34.jpg", "Sana 340.jpg", "Sana 345.jpg", "Sana 346.jpg", "Sana 347.jpg", "Sana 349.jpg", "Sana 35.jpg", "Sana 350.jpg", "Sana 351.jpg", "Sana 36.jpg", "Sana 361.jpg", "Sana 369.jpg", "Sana 37.jpg", "Sana 374.jpg", "Sana 375.jpg", "Sana 377.jpg", "Sana 380.jpg", "Sana 383.jpg", "Sana 384.jpg", "Sana 386.jpg", "Sana 387.jpg", "Sana 388.jpg", "Sana 389.jpg", "Sana 39.jpg", "Sana 392.jpg", "Sana 393.jpg", "Sana 395.jpg", "Sana 399.jpg", "Sana 40.jpg", "Sana 400.jpg", "Sana 401.jpg", "Sana 408.jpg", "Sana 410.jpg", "Sana 411.jpg", "Sana 413.jpg", "Sana 414.jpg", "Sana 419.jpg", "Sana 421.jpg", "Sana 423.jpg", "Sana 425.jpg", "Sana 426.jpg", "Sana 427.jpg", "Sana 43.jpg", "Sana 432.jpg", "Sana 436.jpg", "Sana 438.jpg", "Sana 442.jpg", "Sana 443.jpg", "Sana 446.jpg", "Sana 453.jpg", "Sana 455.jpg", "Sana 457.jpg", "Sana 46.jpg", "Sana 462.jpg", "Sana 464.jpg", "Sana 466.jpg", "Sana 469.jpg", "Sana 473.jpg", "Sana 476.jpg", "Sana 477.jpg", "Sana 48.jpg", "Sana 481.jpg", "Sana 486.jpg", "Sana 488.jpg", "Sana 489.jpg", "Sana 492.jpg", "Sana 496.jpg", "Sana 498.jpg", "Sana 500.jpg", "Sana 507.jpg", "Sana 509.jpg", "Sana 51.jpg", "Sana 511.jpg", "Sana 512.jpg", "Sana 513.jpg", "Sana 514.jpg", "Sana 515.jpg", "Sana 519.jpg", "Sana 52.jpg", "Sana 525.jpg", "Sana 530.jpg", "Sana 531.jpg", "Sana 532.jpg", "Sana 533.jpg", "Sana 536.jpg", "Sana 537.jpg", "Sana 539.jpg", "Sana 548.jpg", "Sana 549.jpg", "Sana 55.jpg", "Sana 551.jpg", "Sana 553.jpg", "Sana 555.jpg", "Sana 556.jpg", "Sana 557.jpg", "Sana 560.jpg", "Sana 565.jpg", "Sana 569.jpg", "Sana 57.jpg", "Sana 572.jpg", "Sana 575.jpg", "Sana 576.jpg", "Sana 577.jpg", "Sana 580.jpg", "Sana 581.jpg", "Sana 582.jpg", "Sana 583.jpg", "Sana 585.jpg", "Sana 586.jpg", "Sana 588.jpg", "Sana 59.jpg", "Sana 591.jpg", "Sana 598.jpg", "Sana 6.jpg", "Sana 60.jpg", "Sana 601.jpg", "Sana 602.jpg", "Sana 608.jpg", "Sana 610.jpg", "Sana 614.jpg", "Sana 617.jpg", "Sana 619.jpg", "Sana 621.jpg", "Sana 622.jpg", "Sana 624.jpg", "Sana 626.jpg", "Sana 629.jpg", "Sana 63.jpg", "Sana 634.jpg", "Sana 637.jpg", "Sana 638.jpg", "Sana 64.jpg", "Sana 643.jpg", "Sana 647.jpg", "Sana 651.jpg", "Sana 653.jpg", "Sana 655.jpg", "Sana 656.jpg", "Sana 657.jpg", "Sana 663.jpg", "Sana 667.jpg", "Sana 668.jpg", "Sana 67.jpg", "Sana 671.jpg", "Sana 672.jpg", "Sana 674.jpg", "Sana 677.jpg", "Sana 682.jpg", "Sana 685.jpg", "Sana 69.jpg", "Sana 691.jpg", "Sana 694.jpg", "Sana 696.jpg", "Sana 7.jpg", "Sana 70.jpg", "Sana 700.jpg", "Sana 701.jpg", "Sana 705.jpg", "Sana 708.jpg", "Sana 715.jpg", "Sana 717.jpg", "Sana 720.jpg", "Sana 723.jpg", "Sana 725.jpg", "Sana 728.jpg", "Sana 73.jpg", "Sana 731.jpg", "Sana 737.jpg", "Sana 739.jpg", "Sana 74.jpg", "Sana 741.jpg", "Sana 743.jpg", "Sana 75.jpg", "Sana 750.jpg", "Sana 752.jpg", "Sana 755.jpg", "Sana 757.jpg", "Sana 762.jpg", "Sana 766.jpg", "Sana 771.jpg", "Sana 775.jpg", "Sana 776.jpg", "Sana 777.jpg", "Sana 779.jpg", "Sana 78.jpg", "Sana 786.jpg", "Sana 788.jpg", "Sana 790.jpg", "Sana 794.jpg", "Sana 795.jpg", "Sana 80.jpg", "Sana 803.jpg", "Sana 808.jpg", "Sana 809.jpg", "Sana 81.jpg", "Sana 813.jpg", "Sana 814.jpg", "Sana 816.jpg", "Sana 817.jpg", "Sana 818.jpg", "Sana 819.jpg", "Sana 820.jpg", "Sana 822.jpg", "Sana 824.jpg", "Sana 827.jpg", "Sana 830.jpg", "Sana 839.jpg", "Sana 84.jpg", "Sana 841.jpg", "Sana 842.jpg", "Sana 843.jpg", "Sana 844.jpg", "Sana 845.jpg", "Sana 846.jpg", "Sana 847.jpg", "Sana 853.jpg", "Sana 857.jpg", "Sana 86.jpg", "Sana 860.jpg", "Sana 861.jpg", "Sana 863.jpg", "Sana 87.jpg", "Sana 873.jpg", "Sana 875.jpg", "Sana 877.jpg", "Sana 886.jpg", "Sana 89.jpg", "Sana 890.jpg", "Sana 892.jpg", "Sana 900.jpg", "Sana 902.jpg", "Sana 903.jpg", "Sana 905.jpg", "Sana 906.jpg", "Sana 91.jpg", "Sana 912.jpg", "Sana 913.jpg", "Sana 917.jpg", "Sana 919.jpg", "Sana 921.jpg", "Sana 928.jpg", "Sana 931.jpg", "Sana 932.jpg", "Sana 934.jpg", "Sana 935.jpg", "Sana 937.jpg", "Sana 938.jpg", "Sana 94.jpg", "Sana 940.jpg", "Sana 943.jpg", "Sana 951.jpg", "Sana 953.jpg", "Sana 956.jpg", "Sana 958.jpg", "Sana 960.jpg", "Sana 966.jpg", "Sana 968.jpg", "Sana 972.jpg", "Sana 976.jpg", "Sana 979.jpg", "Sana 983.jpg", "Sana 99.jpg", "Sana 991.jpg", "Sana 993.jpg"]}, "validation": {"ojo_de_gallo": ["20251129_092009.jpg", "20251129_092333.jpg", "20251129_092735.jpg", "20251129_092925.jpg", "20251129_093509.jpg", "20251129_093848.jpg", "20251129_094057.jpg", "20251129_094345.jpg", "20251129_094716.jpg", "20251129_095022.jpg", "20251129_095346.jpg", "20251129_095547.jpg", "20251129_095626.jpg", "20251129_095703.jpg", "20251129_095949.jpg", "20251129_100037.jpg", "20251129_100540.jpg", "20251129_100648.jpg", "20251129_100822.jpg", "20251129_101002.jpg", "20251129_101038.jpg", "20251129_101044.jpg", "20251129_101243.jpg", "20251129_101733.jpg", "20251129_102732.jpg", "20251129_102831.jpg", "20251129_103110.jpg", "20251129_103945.jpg", "20251129_104403.jpg", "20251129_104738.jpg", "20251129_104907.jpg", "20251129_104918.jpg", "20251129_105430.jpg", "20251129_105445.jpg", "20251129_114552.jpg", "20251129_114708.jpg", "20251129_114714.jpg", "20251129_114939.jpg", "20251129_114946.jpg", "20251129_115257.jpg", "20251129_115342.jpg", "20251129_115513.jpg", "20251129_115618.jpg", "20251129_115807.jpg", "20251129_120332.jpg", "20251129_120401.jpg", "20251129_120633.jpg", "20251129_120920.jpg", "20251129_130229.jpg", "20251129_130232.jpg", "20251129_130322.jpg", "20251129_130533.jpg", "20251129_130657.jpg", "20251129_131129.jpg", "20251129_131356.jpg", "20251129_131402.jpg", "20251129_131638.jpg", "20251129_132216.jpg", "20251129_132305.jpg", "20251129_132632.jpg", "20251129_132910.jpg", "20251129_133030.jpg", "20251129_133527.jpg", "20251129_133840.jpg", "20251129_133933.jpg", "20251129_133943.jpg", "20251129_134656.jpg", "20251129_134708.jpg", "20251129_134753.jpg", "20251129_135137.jpg", "20251129_135531.jpg", "20251129_135606.jpg", "20251129_135624.jpg", "20251129_135927.jpg", "20251129_140749.jpg"], "sana": ["Sana 135.jpg", "Sana 144.jpg", "Sana 155.jpg", "Sana 157.jpg", "Sana 161.jpg", "Sana 19.jpg", "Sana 197.jpg", "Sana 229.jpg", "Sana 239.jpg", "Sana 252.jpg", "Sana 257.jpg", "Sana 297.jpg", "Sana 300.jpg", "Sana 314.jpg", "Sana 316.jpg", "Sana 342.jpg", "Sana 352.jpg", "Sana 355.jpg", "Sana 356.jpg", "Sana 359.jpg", "Sana 379.jpg", "Sana 398.jpg", "Sana 403.jpg", "Sana 428.jpg", "Sana 435.jpg", "Sana 441.jpg", "Sana 467.jpg", "Sana 471.jpg", "Sana 472.jpg", "Sana 50.jpg", "Sana 524.jpg", "Sana 526.jpg", "Sana 528.jpg", "Sana 558.jpg", "Sana 567.jpg", "Sana 574.jpg", "Sana 592.jpg", "Sana 596.jpg", "Sana 611.jpg", "Sana 62.jpg", "Sana 632.jpg", "Sana 633.jpg", "Sana 652.jpg", "Sana 675.jpg", "Sana 684.jpg", "Sana 687.jpg", "Sana 693.jpg", "Sana 697.jpg", "Sana 703.jpg", "Sana 718.jpg", "Sana 72.jpg", "Sana 721.jpg", "Sana 749.jpg", "Sana 765.jpg", "Sana 781.jpg", "Sana 805.jpg", "Sana 826.jpg", "Sana 828.jpg", "Sana 856.jpg", "Sana 866.jpg", "Sana 876.jpg", "Sana 879.jpg", "Sana 882.jpg", "Sana 897.jpg", "Sana 910.jpg", "Sana 936.jpg", "Sana 939.jpg", "Sana 950.jpg", "Sana 957.jpg", "Sana 962.jpg", "Sana 974.jpg", "Sana 980.jpg", "Sana 987.jpg", "Sana 994.jpg"]}, "test": {"ojo_de_gallo": ["20251129_091427.jpg", "20251129_092633.jpg", "20251129_092724.jpg", "20251129_093405.jpg", "20251129_094358.jpg", "20251129_094634.jpg", "20251129_094823.jpg", "20251129_094932.jpg", "20251129_095649.jpg", "20251129_095825.jpg", "20251129_095836.jpg", "20251129_100024.jpg", "20251129_100529.jpg", "20251129_101314.jpg", "20251129_101826.jpg", "20251129_101959.jpg", "20251129_102055.jpg", "20251129_102103.jpg", "20251129_102113.jpg", "20251129_102445.jpg", "20251129_102558.jpg", "20251129_102920.jpg", "20251129_103018.jpg", "20251129_103228.jpg", "20251129_103610.jpg", "20251129_103921.jpg", "20251129_104056.jpg", "20251129_104456.jpg", "20251129_104554.jpg", "20251129_104647.jpg", "20251129_104818.jpg", "20251129_104945.jpg", "20251129_105017.jpg", "20251129_105036.jpg", "20251129_115401.jpg", "20251129_115438.jpg", "20251129_115623.jpg", "20251129_115857.jpg", "20251129_120211.jpg", "20251129_120253.jpg", "20251129_120538.jpg", "20251129_120659.jpg", "20251129_120717.jpg", "20251129_120953.jpg", "20251129_121103.jpg", "20251129_130241.jpg", "20251129_130419.jpg", "20251129_131111.jpg", "20251129_131229.jpg", "20251129_131241.jpg", "20251129_131303.jpg", "20251129_131351.jpg", "20251129_131423.jpg", "20251129_131620.jpg", "20251129_131753.jpg", "20251129_131923.jpg", "20251129_132248.jpg", "20251129_132523.jpg", "20251129_132727.jpg", "20251129_132740.jpg", "20251129_132859.jpg", "20251129_133824.jpg", "20251129_134344.jpg", "20251129_134917.jpg", "20251129_135125.jpg", "20251129_135900.jpg", "20251129_140025.jpg", "20251129_140043.jpg", "20251129_140236.jpg", "20251129_140508.jpg", "20251129_140516.jpg", "20251129_140753.jpg", "20251129_140759.jpg", "20251129_140802.jpg", "20251129_140813.jpg"], "sana": ["Sana 104.jpg", "Sana 119.jpg", "Sana 12.jpg", "Sana 122.jpg", "Sana 123.jpg", "Sana 133.jpg", "Sana 177.jpg", "Sana 187.jpg", "Sana 191.jpg", "Sana 193.jpg", "Sana 198.jpg", "Sana 204.jpg", "Sana 215.jpg", "Sana 23.jpg", "Sana 248.jpg", "Sana 250.jpg", "Sana 275.jpg", "Sana 279.jpg", "Sana 308.jpg", "Sana 309.jpg", "Sana 31.jpg", "Sana 312.jpg", "Sana 320.jpg", "Sana 341.jpg", "Sana 354.jpg", "Sana 360.jpg", "Sana 365.jpg", "Sana 382.jpg", "Sana 417.jpg", "Sana 420.jpg", "Sana 422.jpg", "Sana 431.jpg", "Sana 433.jpg", "Sana 45.jpg", "Sana 452.jpg", "Sana 490.jpg", "Sana 494.jpg", "Sana 516.jpg", "Sana 527.jpg", "Sana 571.jpg", "Sana 594.jpg", "Sana 600.jpg", "Sana 603.jpg", "Sana 615.jpg", "Sana 627.jpg", "Sana 635.jpg", "Sana 636.jpg", "Sana 649.jpg", "Sana 650.jpg", "Sana 660.jpg", "Sana 670.jpg", "Sana 68.jpg", "Sana 689.jpg", "Sana 711.jpg", "Sana 730.jpg", "Sana 732.jpg", "Sana 735.jpg", "Sana 745.jpg", "Sana 761.jpg", "Sana 77.jpg", "Sana 774.jpg", "Sana 79.jpg", "Sana 792.jpg", "Sana 832.jpg", "Sana 837.jpg", "Sana 838.jpg", "Sana 850.jpg", "Sana 869.jpg", "Sana 894.jpg", "Sana 914.jpg", "Sana 933.jpg", "Sana 959.jpg", "Sana 965.jpg", "Sana 97.jpg", "Sana 977.jpg", "Sana 990.jpg"]}}''')

assert os.path.isfile(RUTA_MODELO), f"No encuentro el modelo en {RUTA_MODELO}"
faltantes, filas = [], []
for particion, clases in PARTICION_ORIGINAL.items():
    for clase, archivos in clases.items():
        destino = os.path.join(RUTA_DATASET, particion, clase); os.makedirs(destino, exist_ok=True)
        assert os.path.isdir(RUTA_FOTOS[clase]), f"Falta la carpeta {RUTA_FOTOS[clase]}"
        for nombre in archivos:
            origen = os.path.join(RUTA_FOTOS[clase], nombre)
            if not os.path.isfile(origen):
                faltantes.append(f"{clase}/{nombre}"); continue
            if not os.path.isfile(os.path.join(destino, nombre)):
                shutil.copy2(origen, destino)
        n = len(os.listdir(destino))
        filas.append({"particion": particion, "clase": clase, "imagenes": n, "esperado": len(archivos)})
conteo = pd.DataFrame(filas) if "pd" in globals() else None
print(json.dumps(filas, indent=1, ensure_ascii=False))
if faltantes:
    raise FileNotFoundError(f"Faltan {len(faltantes)} fotos en Drive, por ejemplo: {faltantes[:5]}")
assert all(f["imagenes"] == f["esperado"] for f in filas), "Los conteos no coinciden con la partición original."
print("Partición original reconstruida: los conteos coinciden con la Tabla 4.")

[
 {
  "particion": "train",
  "clase": "ojo_de_gallo",
  "imagenes": 350,
  "esperado": 350
 },
 {
  "particion": "train",
  "clase": "sana",
  "imagenes": 349,
  "esperado": 349
 },
 {
  "particion": "validation",
  "clase": "ojo_de_gallo",
  "imagenes": 75,
  "esperado": 75
 },
 {
  "particion": "validation",
  "clase": "sana",
  "imagenes": 74,
  "esperado": 74
 },
 {
  "particion": "test",
  "clase": "ojo_de_gallo",
  "imagenes": 75,
  "esperado": 75
 },
 {
  "particion": "test",
  "clase": "sana",
  "imagenes": 76,
  "esperado": 76
 }
]
Partición original reconstruida: los conteos coinciden con la Tabla 4.


In [4]:
pd.DataFrame(filas).to_csv(os.path.join(RUTA_SALIDA, "tablas", "conteo_imagenes.csv"), index=False)

## 3. Carga de imágenes
Se cargan igual que en el entrenamiento original: redimensionadas a 224 × 224 con interpolación bilineal.
Etiquetas: `0 = ojo_de_gallo`, `1 = sana` (orden alfabético de las carpetas).

In [5]:
def cargar_particion(particion):
    ds = keras.utils.image_dataset_from_directory(
        os.path.join(RUTA_DATASET, particion), labels="inferred", label_mode="int",
        class_names=CLASES, image_size=(TAM, TAM), interpolation="bilinear",
        batch_size=32, shuffle=False)
    rutas = list(ds.file_paths)
    X, y = [], []
    for xb, yb in ds:
        X.append(xb.numpy().astype("float32")); y.append(yb.numpy())
    return np.concatenate(X), np.concatenate(y), rutas

DATOS = {}
for p in ["train", "validation", "test"]:
    DATOS[p] = cargar_particion(p)
    print(p, DATOS[p][0].shape)

Found 699 files belonging to 2 classes.
train (699, 224, 224, 3)
Found 149 files belonging to 2 classes.
validation (149, 224, 224, 3)
Found 151 files belonging to 2 classes.
test (151, 224, 224, 3)


## 4. Modelo de diagnóstico y línea base (objetivo 1)

In [6]:
modelo = keras.models.load_model(RUTA_MODELO)
base = modelo.get_layer("MobileNetV3Small")
capa_gap = modelo.get_layer("global_average_pooling")
capa_drop = modelo.get_layer("dropout")
capa_salida = modelo.get_layer("clasificacion")

# Modelo de inferencia: el mismo modelo sin las capas de aumento de datos (solo actúan en el entrenamiento).
ent = keras.Input((TAM, TAM, 3), name="imagen")
s = capa_salida(capa_drop(capa_gap(base(ent, training=False)), training=False))
modelo_inf = keras.Model(ent, s, name="diagnostico_ojo_de_gallo")
_x = DATOS["test"][0][:8]
assert np.allclose(modelo.predict(_x, verbose=0), modelo_inf.predict(_x, verbose=0), atol=1e-5)
print("Modelo de inferencia equivalente al original: OK")
print("Parámetros totales:", modelo_inf.count_params())

def salida_cruda(m, X):
    return m.predict(X, batch_size=32, verbose=0).ravel()

# ¿La salida sigmoide representa P(sana) o P(ojo de gallo)? Se determina con el conjunto de VALIDACIÓN.
Xv, yv, _ = DATOS["validation"]
sv = salida_cruda(modelo_inf, Xv)
auc_si_es_sana = roc_auc_score(yv == 1, sv)
SALIDA_ES_P_SANA = auc_si_es_sana >= 0.5
print("La salida del modelo representa:", "P(sana)" if SALIDA_ES_P_SANA else "P(ojo de gallo)",
      f"(AUC de verificación = {max(auc_si_es_sana, 1 - auc_si_es_sana):.4f})")

def p_ojo(m, X):
    s = salida_cruda(m, X)
    return 1.0 - s if SALIDA_ES_P_SANA else s

def wilson(k, n, z=1.959963984540054):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))

def metricas(y, p, umbral=UMBRAL, nombre=""):
    """y: 0=ojo_de_gallo, 1=sana. Clase positiva = ojo de gallo."""
    real = (np.asarray(y) == 0); pred = (np.asarray(p) >= umbral)
    VP = int(np.sum(real & pred)); FN = int(np.sum(real & ~pred))
    VN = int(np.sum(~real & ~pred)); FP = int(np.sum(~real & pred)); n = VP + FN + VN + FP
    sens = VP / (VP + FN) if VP + FN else float("nan")
    esp = VN / (VN + FP) if VN + FP else float("nan")
    prec = VP / (VP + FP) if VP + FP else float("nan")
    f1 = 2 * prec * sens / (prec + sens) if prec + sens else float("nan")
    try:
        auc = roc_auc_score(real, p)
    except ValueError:
        auc = float("nan")
    return {"conjunto": nombre, "n": n, "VP": VP, "FN": FN, "VN": VN, "FP": FP,
            "exactitud": (VP + VN) / n, "exactitud_IC95": wilson(VP + VN, n),
            "sensibilidad": sens, "sensibilidad_IC95": wilson(VP, VP + FN),
            "especificidad": esp, "especificidad_IC95": wilson(VN, VN + FP),
            "precision": prec, "f1": f1, "auc_roc": auc, "umbral": umbral}

def tabla(lista):
    df = pd.DataFrame(lista)
    for c in ["exactitud_IC95", "sensibilidad_IC95", "especificidad_IC95"]:
        if c in df:
            df[c] = df[c].apply(lambda t: f"[{t[0]:.3f}, {t[1]:.3f}]")
    return df

def graficar_matriz(m, titulo, archivo):
    M = np.array([[m["VP"], m["FN"]], [m["FP"], m["VN"]]])
    fig, ax = plt.subplots(figsize=(4.2, 3.8))
    ax.imshow(M, cmap="Greys", vmin=0, vmax=max(M.max(), 1) * 1.6)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(M[i, j]), ha="center", va="center", fontsize=16)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Ojo de gallo", "Sana"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Ojo de gallo", "Sana"])
    ax.set_xlabel("Clase predicha"); ax.set_ylabel("Clase real"); ax.set_title(titulo, fontsize=10)
    fig.tight_layout(); fig.savefig(os.path.join(RUTA_SALIDA, "figuras", archivo), dpi=300); plt.close(fig)

def graficar_roc(curvas, archivo):
    fig, ax = plt.subplots(figsize=(4.6, 4.2))
    for etiqueta, (y, p) in curvas.items():
        fpr, tpr, _ = roc_curve(np.asarray(y) == 0, p)
        ax.plot(fpr, tpr, label=f"{etiqueta} (AUC = {roc_auc_score(np.asarray(y) == 0, p):.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=0.8)
    ax.set_xlabel("Tasa de falsos positivos (1 − especificidad)"); ax.set_ylabel("Sensibilidad")
    ax.legend(fontsize=8, loc="lower right"); fig.tight_layout()
    fig.savefig(os.path.join(RUTA_SALIDA, "figuras", archivo), dpi=300); plt.close(fig)

P = {p: p_ojo(modelo_inf, DATOS[p][0]) for p in ["validation", "test"]}
linea_base = [metricas(DATOS["validation"][1], P["validation"], nombre="Validación"),
              metricas(DATOS["test"][1], P["test"], nombre="Prueba")]
print(tabla(linea_base).to_string(index=False))
graficar_matriz(linea_base[1], "Línea base — conjunto de prueba", "matriz_confusion_linea_base.png")
graficar_roc({"Prueba": (DATOS["test"][1], P["test"])}, "curva_roc_linea_base.png")
guardar_json(linea_base, "linea_base.json")
pd.DataFrame({"imagen": [os.path.basename(r) for r in DATOS["test"][2]],
              "clase_real": [CLASES[i] for i in DATOS["test"][1]],
              "probabilidad_ojo_de_gallo": P["test"]}).to_csv(
    os.path.join(RUTA_SALIDA, "tablas", "predicciones_linea_base_test.csv"), index=False)

Modelo de inferencia equivalente al original: OK
Parámetros totales: 939697
La salida del modelo representa: P(sana) (AUC de verificación = 1.0000)
  conjunto   n  VP  FN  VN  FP  exactitud exactitud_IC95  sensibilidad sensibilidad_IC95  especificidad especificidad_IC95  precision  f1  auc_roc  umbral
Validación 149  75   0  74   0        1.0 [0.975, 1.000]           1.0    [0.951, 1.000]            1.0     [0.951, 1.000]        1.0 1.0      1.0     0.5
    Prueba 151  75   0  76   0        1.0 [0.975, 1.000]           1.0    [0.951, 1.000]            1.0     [0.952, 1.000]        1.0 1.0      1.0     0.5


## 5. Conjunto de prueba depurado (imágenes casi duplicadas)
Para cada imagen de prueba se calcula su distancia de *hash* perceptual a la imagen más parecida de entrenamiento o validación.
Luego se evalúa el modelo quitando las imágenes de prueba que tienen una "casi gemela" (distancia ≤ umbral).

In [7]:
def phash(ruta):
    with Image.open(ruta) as im:
        return imagehash.phash(im.convert("RGB"))

H = {p: [phash(r) for r in DATOS[p][2]] for p in ["train", "validation", "test"]}
ref = H["train"] + H["validation"]
ref_rutas = DATOS["train"][2] + DATOS["validation"][2]
dist_min, gemela = [], []
for h in H["test"]:
    d = [h - r for r in ref]
    k = int(np.argmin(d)); dist_min.append(int(d[k])); gemela.append(ref_rutas[k])
dist_min = np.array(dist_min)
pd.DataFrame({"imagen_prueba": [os.path.basename(r) for r in DATOS["test"][2]],
              "clase": [CLASES[i] for i in DATOS["test"][1]],
              "distancia_minima": dist_min,
              "imagen_mas_parecida": [os.path.relpath(r, RUTA_DATASET) for r in gemela]}).to_csv(
    os.path.join(RUTA_SALIDA, "tablas", "distancias_test_vs_entrenamiento.csv"), index=False)

depurado = []
for u in UMBRALES_DEPURACION:
    keep = dist_min > u
    y_k, p_k = DATOS["test"][1][keep], P["test"][keep]
    m = metricas(y_k, p_k, nombre=f"Prueba depurada (se excluyen distancias ≤ {u})")
    m["excluidas"] = int((~keep).sum())
    depurado.append(m)
print(tabla(depurado).to_string(index=False))
guardar_json(depurado, "prueba_depurada.json")

# Figura para la revisión visual de los 8 pares más parecidos
orden = np.argsort(dist_min)[:8]
fig, axs = plt.subplots(2, 8, figsize=(16, 4.4))
for c, i in enumerate(orden):
    for f, ruta in enumerate([DATOS["test"][2][i], gemela[i]]):
        with Image.open(ruta) as im:
            axs[f, c].imshow(im.convert("RGB").resize((200, 200)))
        axs[f, c].axis("off")
    axs[0, c].set_title(f"d = {dist_min[i]}", fontsize=9)
axs[0, 0].text(-30, 100, "Prueba", rotation=90, va="center"); axs[1, 0].text(-30, 100, "Entren./valid.", rotation=90, va="center")
fig.tight_layout(); fig.savefig(os.path.join(RUTA_SALIDA, "figuras", "pares_mas_parecidos.png"), dpi=200); plt.close(fig)

                                    conjunto   n  VP  FN  VN  FP  exactitud exactitud_IC95  sensibilidad sensibilidad_IC95  especificidad especificidad_IC95  precision  f1  auc_roc  umbral  excluidas
Prueba depurada (se excluyen distancias ≤ 0) 151  75   0  76   0        1.0 [0.975, 1.000]           1.0    [0.951, 1.000]            1.0     [0.952, 1.000]        1.0 1.0      1.0     0.5          0
Prueba depurada (se excluyen distancias ≤ 2) 149  73   0  76   0        1.0 [0.975, 1.000]           1.0    [0.950, 1.000]            1.0     [0.952, 1.000]        1.0 1.0      1.0     0.5          2
Prueba depurada (se excluyen distancias ≤ 4) 129  56   0  73   0        1.0 [0.971, 1.000]           1.0    [0.936, 1.000]            1.0     [0.950, 1.000]        1.0 1.0      1.0     0.5         22


## 6. Grad-CAM: ¿en qué parte de la foto se fija el modelo?
El mapa de calor marca las regiones que más influyen en el diagnóstico.
Además se calcula qué proporción de la "atención" cae en el **centro** de la imagen (60 % × 60 % central, que ocupa el 36 % del área).
Si el modelo mirara al azar, ese valor sería cercano a 0,36; valores más altos indican que se concentra en el centro, donde suele estar la hoja.

In [8]:
def gradcam(X):
    X = tf.convert_to_tensor(X)
    with tf.GradientTape() as tape:
        conv = base(X, training=False)
        tape.watch(conv)
        s = capa_salida(capa_drop(capa_gap(conv), training=False))
        obj = (1.0 - s) if SALIDA_ES_P_SANA else s   # se explica la probabilidad de ojo de gallo
    g = tape.gradient(obj, conv)
    pesos = tf.reduce_mean(g, axis=(1, 2), keepdims=True)
    cam = tf.nn.relu(tf.reduce_sum(pesos * conv, axis=-1)).numpy()
    out = []
    for c in cam:
        c = tf.image.resize(c[..., None], (TAM, TAM)).numpy()[..., 0]
        out.append(c / (c.max() + 1e-8))
    return np.array(out)

def fraccion_central(cam, borde=0.2):
    a = int(TAM * borde); b = TAM - a
    return float(cam[a:b, a:b].sum() / (cam.sum() + 1e-8))

Xt, yt, rt = DATOS["test"]
CAM = np.concatenate([gradcam(Xt[i:i + 16]) for i in range(0, len(Xt), 16)])
fc = np.array([fraccion_central(c) for c in CAM])
res_cam = [{"clase": CLASES[k], "n": int((yt == k).sum()),
            "fraccion_atencion_central_media": float(fc[yt == k].mean()),
            "fraccion_atencion_central_minima": float(fc[yt == k].min())} for k in [0, 1]]
res_cam.append({"clase": "referencia_si_fuera_uniforme", "n": 0,
                "fraccion_atencion_central_media": 0.36, "fraccion_atencion_central_minima": 0.36})
print(pd.DataFrame(res_cam).to_string(index=False))
guardar_json(res_cam, "gradcam_resumen.json")
pd.DataFrame({"imagen": [os.path.basename(r) for r in rt], "clase": [CLASES[i] for i in yt],
              "fraccion_atencion_central": fc}).to_csv(os.path.join(RUTA_SALIDA, "tablas", "gradcam_por_imagen.csv"), index=False)

rng = np.random.RandomState(SEMILLA)
sel = list(rng.choice(np.where(yt == 0)[0], 4, replace=False)) + list(rng.choice(np.where(yt == 1)[0], 4, replace=False))
fig, axs = plt.subplots(2, 8, figsize=(16, 4.6))
for c, i in enumerate(sel):
    axs[0, c].imshow(Xt[i].astype("uint8")); axs[0, c].axis("off")
    axs[0, c].set_title(f"{'Ojo de gallo' if yt[i] == 0 else 'Sana'}\nP(ojo)={P['test'][i]:.2f}", fontsize=8)
    axs[1, c].imshow(Xt[i].astype("uint8")); axs[1, c].imshow(CAM[i], cmap="jet", alpha=0.45); axs[1, c].axis("off")
fig.tight_layout(); fig.savefig(os.path.join(RUTA_SALIDA, "figuras", "gradcam_ejemplos.png"), dpi=200); plt.close(fig)

                       clase  n  fraccion_atencion_central_media  fraccion_atencion_central_minima
                ojo_de_gallo 75                         0.874422                          0.732044
                        sana 76                         0.595795                          0.000000
referencia_si_fuera_uniforme  0                         0.360000                          0.360000


## 7. Modelo verificador: ¿la foto es una hoja de café?
El modelo de diagnóstico solo conoce dos respuestas, así que ante una mano o un objeto igual dice "sana" u "ojo de gallo".
Este segundo modelo decide primero si la imagen es una hoja de café.
- **Positivos (hoja de café):** las 999 imágenes de hojas, con la misma partición del modelo de diagnóstico.
- **Negativos (no es hoja de café):** imágenes públicas de TensorFlow Datasets: manos (`rock_paper_scissors`), objetos y escenas (`imagenette`) y hojas de otros cultivos (`plant_village`).
- **Prueba adicional (opcional):** tus propias fotos en `negativos_propios`.

Se usa la misma receta del modelo de diagnóstico: MobileNetV3Small congelada + capa densa sigmoide.

In [11]:
import tensorflow_datasets as tfds

FUENTES_NEG = [("rock_paper_scissors", "train", "manos"),
               ("imagenette/160px", "train", "objetos y escenas"),
               ("plant_village", "train", "hojas de otros cultivos")]

def cargar_negativos(nombre, split, n):
    ds, info = tfds.load(nombre, split=split, as_supervised=True, with_info=True, shuffle_files=False)
    ds = ds.shuffle(1000, seed=SEMILLA).take(n)
    ims = [tf.image.resize(tf.cast(im, tf.float32), (TAM, TAM), method="bilinear").numpy() for im, _ in ds]
    if ims and ims[0].shape[-1] == 1:
        ims = [np.repeat(i, 3, -1) for i in ims]
    cita = {"nombre": nombre, "homepage": info.homepage, "citation": info.citation, "n_usadas": len(ims)}
    return np.array(ims, dtype="float32"), cita

NEG, CITAS = {}, []
for nombre, split, desc in FUENTES_NEG:
    X_n, cita = cargar_negativos(nombre, split, N_NEGATIVOS_POR_FUENTE)
    cita["descripcion"] = desc; CITAS.append(cita); NEG[desc] = X_n
    print(f"{nombre}: {len(X_n)} imágenes")
guardar_json(CITAS, "fuentes_negativos_citas.json")

# Partición de negativos 70/15/15 por fuente (estratificada por fuente)
neg_part = {"train": [], "validation": [], "test": []}
neg_src = {"train": [], "validation": [], "test": []}
for desc, X_n in NEG.items():
    idx = np.random.RandomState(SEMILLA).permutation(len(X_n))
    a, b = int(0.7 * len(idx)), int(0.85 * len(idx))
    for part, ii in [("train", idx[:a]), ("validation", idx[a:b]), ("test", idx[b:])]:
        neg_part[part].append(X_n[ii]); neg_src[part] += [desc] * len(ii)

def conjunto_verificador(part):
    Xp = DATOS[part][0]; Xn = np.concatenate(neg_part[part])
    X = np.concatenate([Xp, Xn]); y = np.concatenate([np.ones(len(Xp)), np.zeros(len(Xn))]).astype("float32")
    fuente = ["hoja de café"] * len(Xp) + neg_src[part]
    return X, y, fuente

VER = {p: conjunto_verificador(p) for p in ["train", "validation", "test"]}
for p in VER:
    print(p, "hojas de café:", int(VER[p][1].sum()), "| otras imágenes:", int((VER[p][1] == 0).sum()))

aumento = keras.Sequential([keras.layers.RandomFlip("horizontal"), keras.layers.RandomRotation(0.1),
                            keras.layers.RandomZoom(0.1)], name="data_augmentation")
base_v = keras.applications.MobileNetV3Small(input_shape=(TAM, TAM, 3), include_top=False,
                                             weights="imagenet", include_preprocessing=True)
base_v.trainable = False
e = keras.Input((TAM, TAM, 3), name="imagen")
z = base_v(aumento(e), training=False)
z = keras.layers.GlobalAveragePooling2D(name="global_average_pooling")(z)
z = keras.layers.Dropout(0.2, name="dropout")(z)
z = keras.layers.Dense(1, activation="sigmoid", name="es_hoja_de_cafe")(z)
verificador = keras.Model(e, z, name="verificador_hoja_de_cafe")
verificador.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
                    metrics=["accuracy", keras.metrics.AUC(name="auc")])
pesos_clase = {0: len(VER["train"][1]) / (2 * (VER["train"][1] == 0).sum()),
               1: len(VER["train"][1]) / (2 * (VER["train"][1] == 1).sum())}
hist_v = verificador.fit(VER["train"][0], VER["train"][1], validation_data=(VER["validation"][0], VER["validation"][1]),
                         epochs=EPOCAS_VERIFICADOR, batch_size=32, class_weight=pesos_clase, verbose=2,
                         callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)])
verificador.save(os.path.join(RUTA_SALIDA, "modelos", "verificador_hoja_de_cafe.keras"))
guardar_json({k: [float(v) for v in vs] for k, vs in hist_v.history.items()}, "verificador_historial.json")

ent2 = keras.Input((TAM, TAM, 3), name="imagen")
s2 = verificador.get_layer("es_hoja_de_cafe")(verificador.get_layer("dropout")(
    verificador.get_layer("global_average_pooling")(base_v(ent2, training=False)), training=False))
verificador_inf = keras.Model(ent2, s2, name="verificador_inferencia")

def eval_verificador(X, y, fuente, nombre, umbral=UMBRAL_VERIFICADOR):
    p = salida_cruda(verificador_inf, X); acepta = p >= umbral
    filas = [{"conjunto": nombre, "grupo": g, "n": int(np.sum(np.array(fuente) == g)),
              "aceptadas_como_hoja": int(np.sum(acepta[np.array(fuente) == g])),
              "rechazadas": int(np.sum(~acepta[np.array(fuente) == g]))} for g in dict.fromkeys(fuente)]
    hojas = np.array(y) == 1
    resumen = {"conjunto": nombre, "aceptacion_hojas_cafe": float(acepta[hojas].mean()),
               "aceptacion_hojas_IC95": wilson(int(acepta[hojas].sum()), int(hojas.sum())),
               "rechazo_otras_imagenes": float((~acepta[~hojas]).mean()) if (~hojas).any() else float("nan"),
               "rechazo_IC95": wilson(int((~acepta[~hojas]).sum()), int((~hojas).sum())) if (~hojas).any() else None,
               "auc_roc": float(roc_auc_score(hojas, p)) if hojas.any() and (~hojas).any() else float("nan")}
    return filas, resumen, p

f_v, r_v, _ = eval_verificador(*VER["test"], "Prueba")
print(pd.DataFrame(f_v).to_string(index=False)); print(json.dumps(r_v, indent=2, default=float))
resultados_ver = {"prueba_por_grupo": f_v, "prueba_resumen": r_v}

if os.path.isdir(RUTA_NEGATIVOS_PROPIOS):
    rutas_np = sorted([os.path.join(RUTA_NEGATIVOS_PROPIOS, f) for f in os.listdir(RUTA_NEGATIVOS_PROPIOS) if f.lower().endswith(EXT)])
    if rutas_np:
        Xnp = np.array([np.asarray(ImageOps.exif_transpose(Image.open(r)).convert("RGB").resize((TAM, TAM), Image.BILINEAR), dtype="float32") for r in rutas_np])
        pnp = salida_cruda(verificador_inf, Xnp)
        rechazo = float((pnp < UMBRAL_VERIFICADOR).mean())
        resultados_ver["negativos_propios"] = {"n": len(rutas_np), "rechazadas": int((pnp < UMBRAL_VERIFICADOR).sum()),
                                               "tasa_rechazo": rechazo, "IC95": wilson(int((pnp < UMBRAL_VERIFICADOR).sum()), len(rutas_np))}
        pd.DataFrame({"imagen": [os.path.basename(r) for r in rutas_np], "probabilidad_hoja_de_cafe": pnp,
                      "decision": np.where(pnp >= UMBRAL_VERIFICADOR, "aceptada", "rechazada")}).to_csv(
            os.path.join(RUTA_SALIDA, "tablas", "negativos_propios_resultados.csv"), index=False)
        print("Tus fotos propias:", resultados_ver["negativos_propios"])
        # ¿Qué habría dicho el modelo de diagnóstico SIN verificador? (evidencia del problema original)
        pdx = p_ojo(modelo_inf, Xnp)
        resultados_ver["negativos_propios_sin_verificador"] = {
            "diagnosticadas_ojo_de_gallo": int((pdx >= UMBRAL).sum()), "diagnosticadas_sana": int((pdx < UMBRAL).sum())}
        print("Sin verificador, el modelo de diagnóstico habría dicho:", resultados_ver["negativos_propios_sin_verificador"])
else:
    print("No hay carpeta 'negativos_propios' (opcional).")
guardar_json(resultados_ver, "verificador_resultados.json")

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/rock_paper_scissors/incomplete.QC38HY_3.0.0/rock_paper_scissors-train.tfre…

Generating test examples...: 0 examples [00:00, ? examples/s]

... (16 outputs left out)

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/plant_village/incomplete.COSFOG_1.0.2/plant_village-train.tfrecord-[0-9][0…

Dataset plant_village downloaded and prepared to /root/tensorflow_datasets/plant_village/1.0.2. Subsequent calls will reuse this data.
plant_village: 400 imágenes
train hojas de café: 699 | otras imágenes: 840
validation hojas de café: 149 | otras imágenes: 180
test hojas de café: 151 | otras imágenes: 180
4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/15
49/49 - 25s - 506ms/step - accuracy: 0.7817 - auc: 0.8737 - loss: 0.4589 - val_accuracy: 0.9726 - val_auc: 0.9980 - val_loss: 0.2117
Epoch 2/15
49/49 - 3s - 61ms/step - accuracy: 0.9688 - auc: 0.9977 - loss: 0.1603 - val_accuracy: 0.9909 - val_auc: 0.9998 - val_loss: 0.1155
Epoch 3/15
49/49 - 3s - 68ms/step - accuracy: 0.9903 - auc: 0.9998 - loss: 0.0931 - val_accuracy: 0.9878 - val_auc: 0.9998 - val_loss: 0.0884
Epoch 4/15
49/49 - 5s - 102ms/step - accuracy: 0.9909 - auc: 1.0000 - loss: 0.0645 - val_accuracy: 0.9878 - val_auc: 0.9999 - val_loss: 0.0726
Epoch 5/15
49/49 - 3s - 65ms/step - accuracy: 0.9961 - auc: 1.0000 - los

In [12]:
!pip -q install importlib_resources

## 8. Flujo completo en el conjunto de prueba
Simula lo que hará la app con cada foto: **verificador → diagnóstico → resultado** (incluido "no concluyente").

In [13]:
def flujo(X):
    pv = salida_cruda(verificador_inf, X); po = p_ojo(modelo_inf, X)
    dec = np.where(pv < UMBRAL_VERIFICADOR, "rechazada",
                   np.where((po > BANDA_NO_CONCLUYENTE[0]) & (po < BANDA_NO_CONCLUYENTE[1]), "no_concluyente",
                            np.where(po >= UMBRAL, "ojo_de_gallo", "sana")))
    return dec, pv, po

dec, pv, po = flujo(Xt)
real = np.array([CLASES[i] for i in yt])
decididas = np.isin(dec, ["ojo_de_gallo", "sana"])
res_flujo = {"n": len(dec), "rechazadas_por_error": int((dec == "rechazada").sum()),
             "no_concluyentes": int((dec == "no_concluyente").sum()), "decididas": int(decididas.sum()),
             "aciertos_entre_decididas": int((dec[decididas] == real[decididas]).sum()),
             "banda_no_concluyente": BANDA_NO_CONCLUYENTE}
Xn_t = np.concatenate(neg_part["test"]); dn, _, _ = flujo(Xn_t)
res_flujo["otras_imagenes_n"] = len(dn); res_flujo["otras_imagenes_rechazadas"] = int((dn == "rechazada").sum())
print(json.dumps(res_flujo, indent=2, default=float)); guardar_json(res_flujo, "flujo_completo_prueba.json")

{
  "n": 151,
  "rechazadas_por_error": 0,
  "no_concluyentes": 0,
  "decididas": 151,
  "aciertos_entre_decididas": 151,
  "banda_no_concluyente": [
    0.3,
    0.7
  ],
  "otras_imagenes_n": 180,
  "otras_imagenes_rechazadas": 180
}


## 9. Conversión a TensorFlow Lite y efecto de la adaptación (objetivo 2)
Se generan 4 variantes de cada modelo:
- **float32:** sin compresión (referencia).
- **float16:** pesos en 16 bits.
- **dinámica:** pesos en enteros de 8 bits; los cálculos en punto flotante.
- **int8 completa:** pesos y cálculos en enteros de 8 bits (calibrada con 200 imágenes reales de entrenamiento).

Para cada variante se mide el tamaño, el acuerdo con el modelo original, las métricas y si es compatible con XNNPACK, el acelerador que Android usa por defecto.
*El tiempo medido aquí es en el procesador de Colab y solo sirve para comparar variantes: el tiempo real se mide en el Redmi Note 8 (Fase 5).*

In [14]:
from ai_edge_litert.interpreter import Interpreter, OpResolverType

def exportar_variantes(m_inf, nombre, X_cal):
    carpeta_sm = os.path.join("/tmp", f"sm_{nombre}")
    shutil.rmtree(carpeta_sm, ignore_errors=True); m_inf.export(carpeta_sm, verbose=False)
    variantes = {}
    def conv(config):
        c = tf.lite.TFLiteConverter.from_saved_model(carpeta_sm); config(c); return c.convert()
    variantes["float32"] = conv(lambda c: None)
    def f16(c):
        c.optimizations = [tf.lite.Optimize.DEFAULT]; c.target_spec.supported_types = [tf.float16]
    variantes["float16"] = conv(f16)
    variantes["dinamica"] = conv(lambda c: setattr(c, "optimizations", [tf.lite.Optimize.DEFAULT]))
    def i8(c):
        c.optimizations = [tf.lite.Optimize.DEFAULT]
        c.representative_dataset = lambda: ([X_cal[i:i + 1]] for i in range(len(X_cal)))
    variantes["int8_completa"] = conv(i8)
    rutas = {}
    for k, b in variantes.items():
        r = os.path.join(RUTA_SALIDA, "modelos", f"{nombre}_{k}.tflite"); open(r, "wb").write(b); rutas[k] = r
    return rutas

def interprete(ruta):
    try:
        it = Interpreter(model_path=ruta); it.allocate_tensors(); return it, True
    except Exception:
        it = Interpreter(model_path=ruta, experimental_op_resolver_type=OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES)
        it.allocate_tensors(); return it, False

def correr_tflite(ruta, X):
    it, xnn = interprete(ruta); i = it.get_input_details()[0]; o = it.get_output_details()[0]
    salidas, tiempos = [], []
    for k in range(len(X)):
        it.set_tensor(i["index"], X[k:k + 1]); t0 = time.perf_counter(); it.invoke(); tiempos.append(time.perf_counter() - t0)
        salidas.append(it.get_tensor(o["index"]).ravel()[0])
    return np.array(salidas), xnn, float(np.median(tiempos) * 1000)

rng = np.random.RandomState(SEMILLA)
X_cal = DATOS["train"][0][rng.choice(len(DATOS["train"][0]), 200, replace=False)]
rutas_diag = exportar_variantes(modelo_inf, "diagnostico", X_cal)
X_cal_v = VER["train"][0][rng.choice(len(VER["train"][0]), 200, replace=False)]
rutas_ver = exportar_variantes(verificador_inf, "verificador", X_cal_v)

s_ref = salida_cruda(modelo_inf, Xt)
keep4 = dist_min > 4
filas_tfl = []
for k, ruta in rutas_diag.items():
    s, xnn, ms = correr_tflite(ruta, Xt)
    po_k = 1.0 - s if SALIDA_ES_P_SANA else s
    m = metricas(yt, po_k, nombre=f"diagnóstico {k}")
    md = metricas(yt[keep4], po_k[keep4], nombre="")
    filas_tfl.append({"modelo": "diagnóstico", "variante": k, "tamano_MB": os.path.getsize(ruta) / 1e6,
                      "compatible_XNNPACK": xnn, "latencia_mediana_ms_colab": ms,
                      "acuerdo_decision_vs_keras": float(np.mean((s >= 0.5) == (s_ref >= 0.5))),
                      "dif_max_probabilidad": float(np.max(np.abs(s - s_ref))),
                      "exactitud": m["exactitud"], "sensibilidad": m["sensibilidad"], "especificidad": m["especificidad"],
                      "precision": m["precision"], "f1": m["f1"], "auc_roc": m["auc_roc"],
                      "exactitud_prueba_depurada_d4": md["exactitud"]})
Xv_t, yv_t, _ = VER["test"]; sv_ref = salida_cruda(verificador_inf, Xv_t)
for k, ruta in rutas_ver.items():
    s, xnn, ms = correr_tflite(ruta, Xv_t); hojas = yv_t == 1
    filas_tfl.append({"modelo": "verificador", "variante": k, "tamano_MB": os.path.getsize(ruta) / 1e6,
                      "compatible_XNNPACK": xnn, "latencia_mediana_ms_colab": ms,
                      "acuerdo_decision_vs_keras": float(np.mean((s >= UMBRAL_VERIFICADOR) == (sv_ref >= UMBRAL_VERIFICADOR))),
                      "dif_max_probabilidad": float(np.max(np.abs(s - sv_ref))),
                      "aceptacion_hojas_cafe": float((s[hojas] >= UMBRAL_VERIFICADOR).mean()),
                      "rechazo_otras_imagenes": float((s[~hojas] < UMBRAL_VERIFICADOR).mean())})
df_tfl = pd.DataFrame(filas_tfl)
print(df_tfl.to_string(index=False))
df_tfl.to_csv(os.path.join(RUTA_SALIDA, "tablas", "comparacion_variantes_tflite.csv"), index=False)

def recomendar(df, modelo_):
    d = df[(df.modelo == modelo_) & (df.compatible_XNNPACK) & (df.acuerdo_decision_vs_keras >= 0.99)]
    return d.sort_values("tamano_MB").iloc[0]["variante"] if len(d) else "float32"
REC = {"diagnostico": recomendar(df_tfl, "diagnóstico"), "verificador": recomendar(df_tfl, "verificador")}
print("Variante sugerida (la más liviana, compatible con XNNPACK y con ≥ 99 % de acuerdo):", REC)

     modelo      variante  tamano_MB  compatible_XNNPACK  latencia_mediana_ms_colab  acuerdo_decision_vs_keras  dif_max_probabilidad  exactitud  sensibilidad  especificidad  precision       f1  auc_roc  exactitud_prueba_depurada_d4  aceptacion_hojas_cafe  rechazo_otras_imagenes
diagnóstico       float32   3.761596                True                   2.390087                   1.000000              0.000004   1.000000           1.0       1.000000   1.000000 1.000000 1.000000                      1.000000                    NaN                     NaN
diagnóstico       float16   1.954504                True                   2.363963                   1.000000              0.008157   1.000000           1.0       1.000000   1.000000 1.000000 1.000000                      1.000000                    NaN                     NaN
diagnóstico      dinamica   1.138832                True                   4.833278                   0.986755              0.528107   0.986755           1.0      

## 10. Archivos para la aplicación Android y resumen
En `fase2_resultados/android/` quedan los dos modelos sugeridos y el archivo de preprocesamiento que la app debe respetar.

In [15]:
shutil.copy(rutas_diag[REC["diagnostico"]], os.path.join(RUTA_SALIDA, "android", "modelo_diagnostico.tflite"))
shutil.copy(rutas_ver[REC["verificador"]], os.path.join(RUTA_SALIDA, "android", "modelo_verificador.tflite"))
contrato = {
    "entrada": {"alto": TAM, "ancho": TAM, "canales": "RGB", "tipo": "float32", "rango_pixeles": "0 a 255 (sin normalizar: el modelo normaliza internamente)",
                "redimension": "bilineal, imagen completa a 224 x 224"},
    "modelo_verificador": {"archivo": "modelo_verificador.tflite", "salida": "probabilidad de que la imagen sea una hoja de café",
                           "umbral_aceptacion": UMBRAL_VERIFICADOR, "variante": REC["verificador"]},
    "modelo_diagnostico": {"archivo": "modelo_diagnostico.tflite",
                           "salida": "P(sana)" if SALIDA_ES_P_SANA else "P(ojo de gallo)",
                           "probabilidad_ojo_de_gallo": "1 - salida" if SALIDA_ES_P_SANA else "salida",
                           "umbral_ojo_de_gallo": UMBRAL, "banda_no_concluyente": BANDA_NO_CONCLUYENTE, "variante": REC["diagnostico"]},
    "logica": ["Si P(hoja de café) < umbral_aceptacion → 'No parece una hoja de café. Tome otra foto.'",
               "Si P(ojo de gallo) está dentro de la banda_no_concluyente → 'Resultado no concluyente. Tome otra foto.'",
               "Si P(ojo de gallo) ≥ umbral_ojo_de_gallo → 'Ojo de gallo detectado'; si no → 'Hoja sana'"]}
with open(os.path.join(RUTA_SALIDA, "android", "preprocesamiento.json"), "w", encoding="utf-8") as f:
    json.dump(contrato, f, indent=2, ensure_ascii=False)

resumen = {"versiones": VERSIONES, "linea_base": linea_base, "prueba_depurada": depurado, "gradcam": res_cam,
           "verificador": resultados_ver, "flujo_completo": res_flujo, "tflite": filas_tfl, "variantes_sugeridas": REC,
           "fuentes_negativos": [{k: c[k] for k in ("nombre", "homepage", "n_usadas")} for c in CITAS]}
guardar_json(resumen, "resumen_fase2.json")
shutil.make_archive(os.path.join(RUTA_BASE, "fase2_resultados"), "zip", RUTA_SALIDA)
print("LISTO. Todo quedó en:", RUTA_SALIDA, "y en fase2_resultados.zip")
print("Sube el archivo 'tablas/resumen_fase2.json' (o el .zip completo) al chat para escribir los resultados.")

if EN_COLAB:
    from IPython.display import Image as IPImage, display
    for f in ["matriz_confusion_linea_base.png", "curva_roc_linea_base.png", "gradcam_ejemplos.png", "pares_mas_parecidos.png"]:
        display(IPImage(os.path.join(RUTA_SALIDA, "figuras", f), width=700))

Output hidden; open in https://colab.research.google.com to view.